# 07 - Multi-Category Experiments

Extend experiments beyond chairs to **Table** (leg removal) and
**StorageFurniture** (door removal). This demonstrates the
generalizability of the removed-part-aware approach and the
learning-based MLP patch baseline across object categories.

In [1]:
import sys, os, json, gc
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import load_config, ensure_dirs
from src.data.dataset_builder import DatasetBuilder
from src.data.dataset_index import DatasetIndex
from src.experiments.run_batch import run_batch_experiment
from src.experiments.summarize import summarize_results
from src.repair.mlp_patch import MLPPatchGenerator, collect_patch_training_data

print('Imports OK')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Imports OK


In [2]:
cfg = load_config(os.path.join(PROJECT_ROOT, 'configs', 'multi_category.yaml'))

categories = cfg['dataset']['categories']
print('Multi-category experiment setup:')
for cat in categories:
    print(f"  {cat['category']} - remove '{cat['semantic_label']}' ({cat['total_samples']} samples)")

Multi-category experiment setup:
  Chair - remove 'leg' (100 samples)
  Table - remove 'leg' (50 samples)
  StorageFurniture - remove 'door' (50 samples)


## 7.1 Build datasets for each category

For each category, build the paired (complete, damaged) dataset from PartNet.

In [3]:
partnet_root = cfg['paths']['partnet_root']

for cat_cfg in categories:
    category = cat_cfg['category']
    label = cat_cfg['semantic_label']
    n_samples = cat_cfg['total_samples']
    
    data_dir = os.path.join(cfg['paths']['raw_data_dir'], f'{category}_{label}')
    
    # Check if dataset already exists
    index_file = os.path.join(data_dir, 'dataset_index.json')
    if os.path.exists(index_file):
        with open(index_file) as f:
            existing = json.load(f)
        print(f'{category}_{label}: already built ({len(existing)} samples)')
        continue
    
    print(f'\nBuilding {category}_{label} dataset...')
    try:
        builder = DatasetBuilder(
            partnet_root=partnet_root,
            output_dir=data_dir,
            category=category,
            semantic_label=label,
            total_samples=n_samples,
            random_seed=42,
        )
        builder.build()
        print(f'  Done: {category}_{label}')
    except Exception as e:
        print(f'  Failed: {e}')
    gc.collect()

Chair_leg: already built (100 samples)
Table_leg: already built (50 samples)
StorageFurniture_door: already built (50 samples)


## 7.2 Train MLP Patch Generator per category

For each category, train a lightweight MLP to predict patch center offsets.

In [4]:
margin = cfg['repair']['margin']
threshold = cfg['repair']['proximity_threshold']
models_dir = os.path.join(cfg['paths'].get('output_dir', 'outputs'), 'models')
os.makedirs(models_dir, exist_ok=True)

mlp_generators = {}

for cat_cfg in categories:
    category = cat_cfg['category']
    label = cat_cfg['semantic_label']
    cat_key = f'{category}_{label}'
    data_dir = os.path.join(cfg['paths']['raw_data_dir'], cat_key)
    
    if not os.path.exists(os.path.join(data_dir, 'dataset_index.json')):
        print(f'{cat_key}: no dataset, skipping MLP training')
        continue
    
    index = DatasetIndex(data_dir)
    sample_ids = index.sample_ids
    
    # Use 80% for training, 20% for test
    n_train = max(int(len(sample_ids) * 0.8), 3)
    train_ids = sample_ids[:n_train]
    test_ids = sample_ids[n_train:]
    
    print(f'\n{cat_key}: training MLP patch generator...')
    try:
        X_train, y_train = collect_patch_training_data(
            train_ids, data_dir, margin, threshold
        )
        print(f'  Training data: {X_train.shape}')
        
        if len(X_train) < 5:
            print(f'  Too few training samples, skipping')
            continue
        
        gen = MLPPatchGenerator(hidden_sizes=(64, 32, 16), random_state=42)
        gen.fit(X_train, y_train)
        
        train_score = gen.score(X_train, y_train)
        print(f'  Train R²: {train_score:.4f}')
        
        if len(test_ids) > 0:
            X_test, y_test = collect_patch_training_data(
                test_ids, data_dir, margin, threshold
            )
            if len(X_test) > 0:
                test_score = gen.score(X_test, y_test)
                print(f'  Test  R²: {test_score:.4f}')
        
        model_path = os.path.join(models_dir, f'mlp_patch_{cat_key}.pkl')
        gen.save(model_path)
        mlp_generators[cat_key] = gen
        print(f'  Model saved: {model_path}')
    except Exception as e:
        print(f'  MLP training failed: {e}')
    gc.collect()


Chair_leg: training MLP patch generator...
  Training data: (3270, 21)
  Train R²: 0.0000
  Test  R²: 0.0000
  Model saved: D:\MyJupyter\Works\3DPART_v2\outputs\models\mlp_patch_Chair_leg.pkl

Table_leg: training MLP patch generator...
  Training data: (523, 21)
  Train R²: 0.0000
  Test  R²: 0.0000
  Model saved: D:\MyJupyter\Works\3DPART_v2\outputs\models\mlp_patch_Table_leg.pkl

StorageFurniture_door: training MLP patch generator...
  Training data: (819, 21)
  Train R²: 0.0000
  Test  R²: 0.0000
  Model saved: D:\MyJupyter\Works\3DPART_v2\outputs\models\mlp_patch_StorageFurniture_door.pkl


## 7.3 Run experiments for each category

Run all repair methods including MLP Patch + RPA on each category.

In [5]:
all_dfs = {}

for cat_cfg in categories:
    category = cat_cfg['category']
    label = cat_cfg['semantic_label']
    cat_key = f'{category}_{label}'
    data_dir = os.path.join(cfg['paths']['raw_data_dir'], cat_key)
    out_dir = os.path.join(cfg['paths'].get('output_dir', 'outputs'), cat_key)
    
    if not os.path.exists(os.path.join(data_dir, 'dataset_index.json')):
        print(f'{cat_key}: no dataset, skipping')
        continue
    
    index = DatasetIndex(data_dir)
    sample_ids = index.sample_ids
    
    print(f'\nRunning experiments for {cat_key} ({len(sample_ids)} samples)...')
    
    # Get MLP generator if available
    gen = mlp_generators.get(cat_key, None)
    
    try:
        df = run_batch_experiment(
            data_dir=data_dir,
            output_dir=out_dir,
            sample_ids=sample_ids,
            margin=margin,
            proximity_threshold=threshold,
            save_meshes=False,
            include_distance=True,
            include_sota=True,
            mlp_generator=gen,
        )
        df['category'] = category
        df['removed_part'] = label
        all_dfs[cat_key] = df
        print(f'  Done: {len(df)} results')
        
        # Generate per-category tables
        tables = summarize_results(df, out_dir)
        for name, table in tables.items():
            print(f'  {name}: {len(table)} rows')
    except Exception as e:
        print(f'  Failed: {e}')
    gc.collect()

2026-04-15 12:42:35 [INFO] BatchExperiment: Running on 100 samples (SOTA=True, distance=True, mlp=yes)



Running experiments for Chair_leg (100 samples)...


Running experiments: 100%|█████████████████████████████████████████████████████████| 100/100 [1:49:27<00:00, 65.67s/it]
2026-04-15 14:32:02 [INFO] BatchExperiment: Done: 84 success, 16 failed
2026-04-15 14:32:02 [INFO] BatchExperiment: Running on 50 samples (SOTA=True, distance=True, mlp=yes)


  Done: 84 results
  table2_quantitative: 4 rows
  table3_locality: 7 rows
  table4_failures: 10 rows
  table5_target_selection: 2 rows
  table6_sota_comparison: 7 rows
  table7_distance: 7 rows
  table8_mlp_comparison: 3 rows

Running experiments for Table_leg (50 samples)...


Running experiments:   2%|█▏                                                         | 1/50 [04:03<3:18:54, 243.56s/it]2026-04-15 14:36:06 [WARNING] BatchExperiment: Sample 32881: No boundary loops found
2026-04-15 14:36:06 [WARNING] BatchExperiment: Sample 29101: No boundary loops found
Running experiments:  72%|███████████████████████████████████████████▉                 | 36/50 [45:01<16:57, 72.70s/it]2026-04-15 15:17:04 [WARNING] BatchExperiment: Sample 28960: No boundary loops found
2026-04-15 15:17:04 [WARNING] BatchExperiment: Sample 27825: No boundary loops found
Running experiments:  78%|███████████████████████████████████████████████▌             | 39/50 [45:50<07:33, 41.26s/it]2026-04-15 15:17:53 [WARNING] BatchExperiment: Sample 25161: No boundary loops found
2026-04-15 15:17:53 [WARNING] BatchExperiment: Sample 28779: No boundary loops found
Running experiments: 100%|███████████████████████████████████████████████████████████| 50/50 [1:00:37<00:00, 72.74s/it]
2026-04-15 15

  Done: 28 results
  table2_quantitative: 4 rows
  table3_locality: 7 rows
  table4_failures: 10 rows
  table5_target_selection: 2 rows
  table6_sota_comparison: 7 rows
  table7_distance: 7 rows
  table8_mlp_comparison: 3 rows

Running experiments for StorageFurniture_door (50 samples)...


Running experiments:   6%|███▋                                                          | 3/50 [04:57<58:34, 74.77s/it]2026-04-15 15:37:37 [WARNING] BatchExperiment: Sample 47987: No boundary loops found
2026-04-15 15:37:37 [WARNING] BatchExperiment: Sample 47531: No boundary loops found
Running experiments:  20%|████████████▏                                                | 10/50 [17:00<52:11, 78.29s/it]2026-04-15 15:49:40 [WARNING] BatchExperiment: Sample 41129: No boundary loops found
2026-04-15 15:49:40 [WARNING] BatchExperiment: Sample 47654: No boundary loops found
Running experiments:  38%|██████████████████████                                    | 19/50 [39:42<1:00:35, 117.29s/it]2026-04-15 16:12:22 [WARNING] BatchExperiment: Sample 48343: No boundary loops found
2026-04-15 16:12:22 [WARNING] BatchExperiment: Sample 47595: No boundary loops found
2026-04-15 16:12:22 [WARNING] BatchExperiment: Sample 45843: No boundary loops found
Running experiments:  60%|██████████████████████

  Done: 32 results
  table2_quantitative: 4 rows
  table3_locality: 7 rows
  table4_failures: 10 rows
  table5_target_selection: 2 rows
  table6_sota_comparison: 7 rows
  table7_distance: 7 rows
  table8_mlp_comparison: 3 rows


## 7.4 Cross-category comparison

In [6]:
if len(all_dfs) > 0:
    methods = {
        'Center-fan + RPA': 'center_fan',
        'Planar + RPA': 'planar_removed_part_aware',
        'MLP Patch + RPA': 'mlp_patch_rpa',
        'Adv-front + RPA': 'advancing_front_rpa',
        'Planar + LH-only': 'planar_largest_hole_only',
    }
    
    metrics_to_show = {
        'Locality': 'locality_ratio',
        'Avg Quality': 'avg_new_face_quality',
        'Residual': 'target_loop_length_after',
        'CD': 'chamfer_distance',
    }
    
    print('\n=== Cross-category summary ===')
    for cat_key, cat_df in all_dfs.items():
        print(f'\n--- {cat_key} ---')
        for method_name, prefix in methods.items():
            loc_col = f'{prefix}/locality_ratio'
            qual_col = f'{prefix}/avg_new_face_quality'
            res_col = f'{prefix}/target_loop_length_after'
            cd_col = f'{prefix}/chamfer_distance'
            
            loc = cat_df[loc_col].mean() if loc_col in cat_df.columns else float('nan')
            qual = cat_df[qual_col].mean() if qual_col in cat_df.columns else float('nan')
            res = cat_df[res_col].mean() if res_col in cat_df.columns else float('nan')
            cd = cat_df[cd_col].mean() * 1000 if cd_col in cat_df.columns else float('nan')
            
            print(f'  {method_name:25s}  Loc={loc:.4f}  Qual={qual:.4f}  Res={res:.4f}  CD={cd:.2f}')


=== Cross-category summary ===

--- Chair_leg ---
  Center-fan + RPA           Loc=0.6656  Qual=0.2671  Res=0.0000  CD=81.50
  Planar + RPA               Loc=0.6710  Qual=0.4341  Res=1.9384  CD=81.18
  MLP Patch + RPA            Loc=0.6336  Qual=0.2226  Res=0.0000  CD=83.68
  Adv-front + RPA            Loc=0.7174  Qual=0.4406  Res=0.0000  CD=79.57
  Planar + LH-only           Loc=0.5717  Qual=0.4122  Res=38.2569  CD=79.24

--- Table_leg ---
  Center-fan + RPA           Loc=0.8511  Qual=0.3136  Res=0.0000  CD=75.41
  Planar + RPA               Loc=0.8654  Qual=0.4812  Res=0.7670  CD=74.55
  MLP Patch + RPA            Loc=0.8334  Qual=0.2528  Res=0.0000  CD=78.71
  Adv-front + RPA            Loc=0.9185  Qual=0.5432  Res=0.0000  CD=66.91
  Planar + LH-only           Loc=0.6981  Qual=0.4639  Res=32.8290  CD=73.65

--- StorageFurniture_door ---
  Center-fan + RPA           Loc=0.1614  Qual=0.1135  Res=0.0000  CD=37.93
  Planar + RPA               Loc=0.1601  Qual=0.3799  Res=1.5902  CD=37.

In [7]:
# Cross-category comparison figure
if len(all_dfs) > 1:
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    
    cat_names = list(all_dfs.keys())
    method_list = ['center_fan', 'planar_removed_part_aware', 'mlp_patch_rpa']
    method_labels = ['Center-fan', 'Planar+RPA', 'MLP Patch']
    colors = ['#4ECDC4', '#45B7D1', '#FF6B6B']
    
    metric_list = [
        ('locality_ratio', 'Locality'),
        ('avg_new_face_quality', 'Avg Quality'),
        ('target_loop_length_after', 'Residual'),
        ('chamfer_distance', 'CD (×10³)'),
    ]
    
    for ax_idx, (metric_col, metric_label) in enumerate(metric_list):
        ax = axes[ax_idx]
        x = np.arange(len(cat_names))
        width = 0.25
        
        for m_idx, (prefix, m_label) in enumerate(zip(method_list, method_labels)):
            vals = []
            for cat_key in cat_names:
                col = f'{prefix}/{metric_col}'
                if col in all_dfs[cat_key].columns:
                    v = all_dfs[cat_key][col].dropna().mean()
                    if 'CD' in metric_label:
                        v *= 1000
                    vals.append(v)
                else:
                    vals.append(0)
            ax.bar(x + m_idx * width, vals, width, label=m_label, color=colors[m_idx])
        
        ax.set_xticks(x + width)
        ax.set_xticklabels([k.replace('_', '\n') for k in cat_names], fontsize=9)
        ax.set_title(metric_label)
        if ax_idx == 0:
            ax.legend(fontsize=8)
    
    plt.suptitle('Cross-Category Comparison', fontsize=13)
    plt.tight_layout()
    
    fig_dir = os.path.join(cfg['paths'].get('output_dir', 'outputs'), 'figures')
    os.makedirs(fig_dir, exist_ok=True)
    plt.savefig(os.path.join(fig_dir, 'cross_category_comparison.pdf'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(fig_dir, 'cross_category_comparison.png'), dpi=150, bbox_inches='tight')
    print('Cross-category figure saved.')
    plt.close()

Cross-category figure saved.


## 7.5 Combined multi-category table for paper

Generate a LaTeX-ready table combining results from all categories.

In [8]:
if len(all_dfs) > 0:
    # Build combined table
    rows = []
    methods = {
        'Center-fan + RPA': 'center_fan',
        'Planar + RPA': 'planar_removed_part_aware',
        'MLP Patch + RPA': 'mlp_patch_rpa',
        'Adv-front + RPA': 'advancing_front_rpa',
    }
    
    for cat_key, cat_df in all_dfs.items():
        for method_name, prefix in methods.items():
            row = {'Category': cat_key, 'Method': method_name}
            for metric, col_suffix in [
                ('Residual', 'target_loop_length_after'),
                ('Quality', 'avg_new_face_quality'),
                ('Locality', 'locality_ratio'),
                ('CD×10³', 'chamfer_distance'),
            ]:
                col = f'{prefix}/{col_suffix}'
                if col in cat_df.columns:
                    v = cat_df[col].dropna().mean()
                    if 'CD' in metric:
                        v *= 1000
                    row[metric] = round(v, 4)
                else:
                    row[metric] = float('nan')
            rows.append(row)
    
    combined_df = pd.DataFrame(rows)
    
    tables_dir = os.path.join(cfg['paths'].get('output_dir', 'outputs'), 'tables')
    os.makedirs(tables_dir, exist_ok=True)
    combined_df.to_csv(os.path.join(tables_dir, 'table_multi_category.csv'), index=False)
    
    print('\n=== Multi-Category Results ===')
    print(combined_df.to_string(index=False))


=== Multi-Category Results ===
             Category           Method  Residual  Quality  Locality  CD×10³
            Chair_leg Center-fan + RPA    0.0000   0.2671    0.6656 81.4953
            Chair_leg     Planar + RPA    1.9384   0.4341    0.6710 81.1804
            Chair_leg  MLP Patch + RPA    0.0000   0.2226    0.6336 83.6753
            Chair_leg  Adv-front + RPA    0.0000   0.4406    0.7174 79.5681
            Table_leg Center-fan + RPA    0.0000   0.3136    0.8511 75.4063
            Table_leg     Planar + RPA    0.7670   0.4812    0.8654 74.5542
            Table_leg  MLP Patch + RPA    0.0000   0.2528    0.8334 78.7059
            Table_leg  Adv-front + RPA    0.0000   0.5432    0.9185 66.9108
StorageFurniture_door Center-fan + RPA    0.0000   0.1135    0.1614 37.9287
StorageFurniture_door     Planar + RPA    1.5902   0.3799    0.1601 37.5346
StorageFurniture_door  MLP Patch + RPA    0.0000   0.0948    0.0792 49.7862
StorageFurniture_door  Adv-front + RPA    0.0000   0.380

## 7.6 Summary

The multi-category experiments demonstrate that:
1. The removed-part-aware target selection generalizes across categories
2. MLP Patch + RPA provides consistent improvement over geometric baselines
3. The evaluation protocol (closure, quality, locality, distance) is
   applicable beyond the chair category